# Доп. задание: Identification Rate Metric (TPR@FPR) — Kaggle

Задание (3 балла): реализовать метрику Identification Rate (TPR@FPR) — она, в отличие от accuracy,
измеряет качество модели на людях, которых не было в обучающей выборке.



In [1]:
import os

TASK1_INPUT_DIR = '/kaggle/input/notebooks/iuliiaburmistrova/notebook0e9be24a8a'
TASK2_INPUT_DIR = '/kaggle/input/notebooks/iuliiaburmistrova/notebook98e2177099'



In [2]:
import itertools
import shutil

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## 1. Реализация метрики (шаблон из задания)

In [3]:
def compute_embeddings(model, images_list):
    """compute embeddings from the trained model for list of images."""
    embeddings = []
    model.eval()
    with torch.no_grad():
        for path in images_list:
            img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (112, 112))
            tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            tensor = ((tensor - 0.5) / 0.5).unsqueeze(0).to(DEVICE)
            emb = model(tensor)
            emb = F.normalize(emb, dim=1).cpu().numpy()[0]
            embeddings.append(emb)
    return embeddings


def compute_cosine_query_pos(query_dict, query_img_names, query_embeddings):
    """cosine similarities between positive pairs from query (stage 1)."""
    name_to_emb = {name: np.asarray(emb) for name, emb in zip(query_img_names, query_embeddings)}
    similarities = []
    for class_id, names in query_dict.items():
        for a, b in itertools.combinations(names, 2):
            emb_a, emb_b = name_to_emb[a], name_to_emb[b]
            sim = float(np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b)))
            similarities.append(sim)
    return similarities


def compute_cosine_query_neg(query_dict, query_img_names, query_embeddings):
    """cosine similarities between negative pairs from query (stage 2)."""
    name_to_emb = {name: np.asarray(emb) for name, emb in zip(query_img_names, query_embeddings)}
    class_ids = list(query_dict.keys())
    similarities = []
    for i, cid_a in enumerate(class_ids):
        for cid_b in class_ids[i + 1:]:
            for a in query_dict[cid_a]:
                for b in query_dict[cid_b]:
                    emb_a, emb_b = name_to_emb[a], name_to_emb[b]
                    sim = float(np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b)))
                    similarities.append(sim)
    return similarities


def compute_cosine_query_distractors(query_embeddings, distractors_embeddings):
    """cosine similarities between every (query, distractor) pair (stage 3)."""
    similarities = []
    for q_emb in query_embeddings:
        q = np.asarray(q_emb)
        for d_emb in distractors_embeddings:
            d = np.asarray(d_emb)
            sim = float(np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d)))
            similarities.append(sim)
    return similarities


def compute_ir(cosine_query_pos, cosine_query_neg, cosine_query_distractors, fpr=0.1):
    """compute identification rate (threshold, TPR) at given FPR."""
    import math

    false_pairs = list(cosine_query_neg) + list(cosine_query_distractors)
    false_pairs_sorted = sorted(false_pairs, reverse=True)

    n_allowed = max(1, math.ceil(fpr * len(false_pairs_sorted)))
    threshold = false_pairs_sorted[n_allowed - 1]

    pos = np.asarray(cosine_query_pos)
    tpr = float(np.mean(pos >= threshold))
    return threshold, tpr


Проверка на контрольных данных из шаблона задания:

In [4]:
test_query_dict = {
    2876: ['1.jpg', '2.jpg', '3.jpg'],
    5674: ['5.jpg'],
    864:  ['9.jpg', '10.jpg'],
}
test_query_img_names = ['1.jpg', '2.jpg', '3.jpg', '5.jpg', '9.jpg', '10.jpg']
test_query_embeddings = [
                    [1.56, 6.45,  -7.68],
                    [-1.1 , 6.11,  -3.0],
                    [-0.06,-0.98,-1.29],
                    [8.56, 1.45,  1.11],
                    [0.7,  1.1,   -7.56],
                    [0.05, 0.9,   -2.56],
]
test_distractors_img_names = ['11.jpg', '12.jpg', '13.jpg', '14.jpg', '15.jpg']
test_distractors_embeddings = [
                    [0.12, -3.23, -5.55],
                    [-1,   -0.01, 1.22],
                    [0.06, -0.23, 1.34],
                    [-6.6, 1.45,  -1.45],
                    [0.89,  1.98, 1.45],
]

test_cosine_query_pos = compute_cosine_query_pos(test_query_dict, test_query_img_names, test_query_embeddings)
test_cosine_query_neg = compute_cosine_query_neg(test_query_dict, test_query_img_names, test_query_embeddings)
test_cosine_query_distractors = compute_cosine_query_distractors(test_query_embeddings, test_distractors_embeddings)

true_cosine_query_pos = [0.8678237233650096, 0.21226104378511604, -0.18355866977496182, 0.9787437979250561]
assert np.allclose(sorted(test_cosine_query_pos), sorted(true_cosine_query_pos)), "compute_cosine_query_pos: ошибка"

true_cosine_query_neg = [0.15963231223161822, 0.8507997093616965, 0.9272761484302097, -0.0643994061127092,
                         0.5412660901220571, 0.701307100338029, -0.2372575528216902, 0.6941032794522218,
                         0.549425446066643, -0.011982733001947084, -0.0466679194884999]
assert np.allclose(sorted(test_cosine_query_neg), sorted(true_cosine_query_neg)), "compute_cosine_query_neg: ошибка"

test_thr, test_tpr = [], []
for fpr in [0.5, 0.3, 0.1]:
    x, y = compute_ir(test_cosine_query_pos, test_cosine_query_neg, test_cosine_query_distractors, fpr=fpr)
    test_thr.append(x); test_tpr.append(y)

true_thr = [-0.011982733001947084, 0.3371426578637511, 0.701307100338029]
assert np.allclose(np.array(test_thr), np.array(true_thr)), "compute_ir: ошибка в threshold"

true_tpr = [0.75, 0.5, 0.5]
assert np.allclose(np.array(test_tpr), np.array(true_tpr)), "compute_ir: ошибка в tpr"

print('Все тесты пройдены')


Все тесты пройдены


## 2. Скачивание CelebA + архитектура Hourglass (для align новых личностей)

`landmarks`/`identity` берём из output Task 1 (не качаем повторно), `bbox`/`attr` — из датасета Kaggle.

In [5]:
!pip install -q kagglehub

import kagglehub

KAGGLE_DATASET = 'kevinpatel04/celeba-original-wild-images'
kaggle_path = kagglehub.dataset_download(KAGGLE_DATASET)
print('Датасет скачан в:', kaggle_path)

BBOX_FILE = os.path.join(kaggle_path, 'list_bbox_celeba.csv')
ATTR_FILE = os.path.join(kaggle_path, 'list_attr_celeba.csv')
LANDMARKS_FILE = os.path.join(TASK1_INPUT_DIR, 'data', 'celeba', 'list_landmarks_celeba.txt')
IDENTITY_FILE = os.path.join(TASK1_INPUT_DIR, 'data', 'celeba', 'identity_CelebA.txt')

assert os.path.exists(LANDMARKS_FILE) and os.path.exists(IDENTITY_FILE), \
    'landmarks/identity не найдены в output Task 1 - проверьте TASK1_INPUT_DIR'


def build_image_index(kaggle_path, n_parts=21):
    index = {}
    for i in range(1, n_parts + 1):
        part_dir = os.path.join(kaggle_path, f'Part {i}', f'Part {i}')
        if not os.path.isdir(part_dir):
            continue
        for fname in os.listdir(part_dir):
            index[fname] = os.path.join(part_dir, fname)
    return index

IMAGE_INDEX = build_image_index(kaggle_path)
print('Найдено изображений в индексе:', len(IMAGE_INDEX))


def load_celeba_annotations():
    bbox = pd.read_csv(BBOX_FILE)
    attr = pd.read_csv(ATTR_FILE)
    landmarks = pd.read_csv(LANDMARKS_FILE, sep=r'\s+', skiprows=1)
    landmarks = landmarks.reset_index().rename(columns={'index': 'image_id'})
    identity = pd.read_csv(IDENTITY_FILE, sep=r'\s+', header=None, names=['image_id', 'identity'])
    return bbox.merge(landmarks, on='image_id').merge(attr, on='image_id').merge(identity, on='image_id')


df = load_celeba_annotations()
print('Аннотации CelebA загружены:', len(df), 'строк')


Датасет скачан в: /kaggle/input/datasets/kevinpatel04/celeba-original-wild-images
Найдено изображений в индексе: 202599
Аннотации CelebA загружены: 202599 строк


In [6]:
# --- Архитектура Stacked Hourglass (Task 1) - нужна только для align новых личностей ---

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels // 2, 1)
        self.bn1 = nn.BatchNorm2d(out_channels // 2)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels // 2, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels // 2)
        self.conv3 = nn.Conv2d(out_channels // 2, out_channels, 1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        return self.relu(x + residual)


class Hourglass(nn.Module):
    def __init__(self, depth, num_channels):
        super().__init__()
        self.up1 = ResidualBlock(num_channels, num_channels)
        self.pool = nn.MaxPool2d(2)
        self.low1 = ResidualBlock(num_channels, num_channels)
        self.low2 = Hourglass(depth - 1, num_channels) if depth > 1 else ResidualBlock(num_channels, num_channels)
        self.low3 = ResidualBlock(num_channels, num_channels)
        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, x):
        up1 = self.up1(x)
        pool = self.pool(x)
        low1 = self.low1(pool)
        low2 = self.low2(low1)
        low3 = self.low3(low2)
        up2 = self.up2(low3)
        return up1 + up2


class StackedHourglass(nn.Module):
    def __init__(self, num_stacks=4, num_channels=256, num_landmarks=5, hg_depth=4):
        super().__init__()
        self.num_stacks = num_stacks
        self.pre = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            ResidualBlock(64, 128), nn.MaxPool2d(2), ResidualBlock(128, 128), ResidualBlock(128, num_channels),
        )
        self.hourglasses = nn.ModuleList([Hourglass(hg_depth, num_channels) for _ in range(num_stacks)])
        self.features = nn.ModuleList([
            nn.Sequential(ResidualBlock(num_channels, num_channels), nn.Conv2d(num_channels, num_channels, 1),
                          nn.BatchNorm2d(num_channels), nn.ReLU(inplace=True))
            for _ in range(num_stacks)
        ])
        self.heatmap_heads = nn.ModuleList([nn.Conv2d(num_channels, num_landmarks, 1) for _ in range(num_stacks)])
        self.merge_features = nn.ModuleList([nn.Conv2d(num_channels, num_channels, 1) for _ in range(num_stacks - 1)])
        self.merge_heatmaps = nn.ModuleList([nn.Conv2d(num_landmarks, num_channels, 1) for _ in range(num_stacks - 1)])

    def forward(self, x):
        x = self.pre(x)
        outputs = []
        for i in range(self.num_stacks):
            hg_out = self.hourglasses[i](x)
            feat = self.features[i](hg_out)
            heatmap = self.heatmap_heads[i](feat)
            outputs.append(heatmap)
            if i < self.num_stacks - 1:
                x = x + self.merge_features[i](feat) + self.merge_heatmaps[i](heatmap)
        return outputs


def heatmaps_to_landmarks(heatmaps):
    landmarks = []
    for hm in heatmaps:
        y, x = np.unravel_index(np.argmax(hm), hm.shape)
        landmarks.append((float(x), float(y)))
    return np.array(landmarks, dtype=np.float32)


@torch.no_grad()
def predict_landmarks(model, image_rgb, img_size=256, heatmap_size=64, device=DEVICE):
    model.eval()
    resized = cv2.resize(image_rgb, (img_size, img_size))
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    tensor = torch.from_numpy(resized).permute(2, 0, 1).float() / 255.0
    tensor = normalize(tensor).unsqueeze(0).to(device)
    heatmaps = model(tensor)[-1][0].cpu().numpy()
    return resized, heatmaps_to_landmarks(heatmaps) * (img_size / heatmap_size)


REFERENCE_LANDMARKS_112 = np.array([
    [38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
    [41.5493, 92.3655], [70.7299, 92.2041],
], dtype=np.float32)


def align_face(image_rgb, landmarks, output_size=112):
    ref = REFERENCE_LANDMARKS_112.copy() * (output_size / 112.0)
    src = np.asarray(landmarks, dtype=np.float32)
    M, _ = cv2.estimateAffinePartial2D(src, ref, method=cv2.LMEDS)
    if M is None:
        return cv2.resize(image_rgb, (output_size, output_size))
    return cv2.warpAffine(image_rgb, M, (output_size, output_size), borderValue=(0, 0, 0))


HOURGLASS_WEIGHTS = os.path.join(TASK1_INPUT_DIR, 'checkpoints_hourglass', 'best.pth')
hg_model = StackedHourglass(num_stacks=4, num_channels=256, num_landmarks=5, hg_depth=4)
hg_model.load_state_dict(torch.load(HOURGLASS_WEIGHTS, map_location=DEVICE))
hg_model = hg_model.to(DEVICE).eval()
print('Hourglass из Task 1 загружен')


BBOX_MARGIN = 0.35


def crop_and_save(df, image_index, out_dir, margin=BBOX_MARGIN):
    os.makedirs(os.path.join(out_dir, 'images'), exist_ok=True)
    records, skipped = [], 0
    for _, row in df.iterrows():
        img_path = image_index.get(row['image_id'])
        if img_path is None:
            skipped += 1
            continue
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]
        x1, y1, bw, bh = row['x_1'], row['y_1'], row['width'], row['height']
        mx, my = bw * margin, bh * margin
        x1c, y1c = max(0, int(x1 - mx)), max(0, int(y1 - my))
        x2c, y2c = min(w, int(x1 + bw + mx)), min(h, int(y1 + bh + my))
        crop = img[y1c:y2c, x1c:x2c]
        if crop.size == 0:
            skipped += 1
            continue
        cv2.imwrite(os.path.join(out_dir, 'images', row['image_id']), crop)
        records.append({'image_id': row['image_id'], 'identity': row['identity']})
    if skipped:
        print(f'Пропущено {skipped} изображений')
    result = pd.DataFrame(records)
    result.to_csv(os.path.join(out_dir, 'selected_dataset.csv'), index=False)
    return result


@torch.no_grad()
def build_aligned_dataset(model, csv_file, img_dir, output_dir,
                           img_size=256, heatmap_size=64, output_size=112, device=DEVICE):
    model.eval()
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    df_local = pd.read_csv(csv_file)
    records = []
    for _, row in df_local.iterrows():
        img = cv2.imread(os.path.join(img_dir, row['image_id']))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        resized, landmarks = predict_landmarks(model, img_rgb, img_size, heatmap_size, device)
        aligned = align_face(resized, landmarks, output_size)
        cv2.imwrite(os.path.join(output_dir, 'images', row['image_id']), cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR))
        records.append({'image_id': row['image_id'], 'identity': row['identity']})
    result_df = pd.DataFrame(records)
    result_df.to_csv(os.path.join(output_dir, 'aligned_dataset.csv'), index=False)
    return result_df

print('Функции подготовки данных готовы')


Hourglass из Task 1 загружен
Функции подготовки данных готовы


## 3. Формирование query и distractors — личности, НЕ пересекающиеся с обучением Task 2

In [10]:
RECOG_TRAIN_IDENTITIES_CSV = os.path.join(TASK2_INPUT_DIR, 'data', 'prepared_recog', 'selected_dataset.csv')


def prepare_query_distractors_identities(df, used_identities, n_query_identities=15,
                                          n_distractors_identities=60,
                                          photos_per_query_identity=4,
                                          seed=SEED):
    """Отбирает НОВЫЕ (не пересекающиеся с used_identities) личности CelebA для query/distractors."""
    remaining = df[~df['identity'].isin(used_identities)]
    counts = remaining['identity'].value_counts()
    eligible = counts[counts >= photos_per_query_identity].index.tolist()

    rng = np.random.RandomState(seed)
    chosen = list(rng.choice(eligible, size=n_query_identities + n_distractors_identities, replace=False))
    query_ids = chosen[:n_query_identities]
    distractor_ids = chosen[n_query_identities:]

    query_df = remaining[remaining['identity'].isin(query_ids)].groupby('identity', group_keys=False) \
                        .apply(lambda g: g.sample(min(len(g), photos_per_query_identity), random_state=seed))
    distractor_df = remaining[remaining['identity'].isin(distractor_ids)].groupby('identity', group_keys=False) \
                        .apply(lambda g: g.sample(1, random_state=seed))

    return query_df.reset_index(drop=True), distractor_df.reset_index(drop=True)


used_identities = pd.read_csv(RECOG_TRAIN_IDENTITIES_CSV)['identity'].unique().tolist()
query_df, distractor_df = prepare_query_distractors_identities(df, used_identities)
print(f'Query: {query_df["identity"].nunique()} личностей, {len(query_df)} фото')
print(f'Distractors: {distractor_df["identity"].nunique()} личностей, {len(distractor_df)} фото')

query_prepared = crop_and_save(query_df, IMAGE_INDEX, out_dir='data/prepared_query')
distractor_prepared = crop_and_save(distractor_df, IMAGE_INDEX, out_dir='data/prepared_distractors')

build_aligned_dataset(hg_model, 'data/prepared_query/selected_dataset.csv',
                       'data/prepared_query/images', 'data/aligned_query')
build_aligned_dataset(hg_model, 'data/prepared_distractors/selected_dataset.csv',
                       'data/prepared_distractors/images', 'data/aligned_distractors')

if kaggle_path and os.path.exists(kaggle_path):
    shutil.rmtree(kaggle_path, ignore_errors=True)
    print('Сырые изображения удалены')


/tmp/ipykernel_58/2079502184.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), photos_per_query_identity), random_state=seed))
/tmp/ipykernel_58/2079502184.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(1, random_state=seed))


Query: 15 личностей, 60 фото
Distractors: 60 личностей, 60 фото
Сырые изображения удалены


## 4. Подсчёт TPR@FPR для CE и ArcFace моделей из Task 2

In [11]:
EMBEDDING_SIZE = 512


def build_backbone(embedding_size=EMBEDDING_SIZE):
    backbone = models.efficientnet_b0(weights=None)
    in_features = backbone.classifier[1].in_features
    backbone.classifier = nn.Linear(in_features, embedding_size)
    return backbone


def load_backbone_only(checkpoint_path):
    backbone = build_backbone().to(DEVICE)
    full_state = torch.load(checkpoint_path, map_location=DEVICE)
    backbone_state = {k.replace('backbone.', ''): v for k, v in full_state.items() if k.startswith('backbone.')}
    backbone.load_state_dict(backbone_state)
    backbone.eval()
    return backbone


ce_backbone = load_backbone_only(os.path.join(TASK2_INPUT_DIR, 'checkpoints_recognition', 'ce_baseline_best.pth'))
arcface_backbone = load_backbone_only(os.path.join(TASK2_INPUT_DIR, 'checkpoints_recognition', 'arcface_best.pth'))
print('CE и ArcFace backbone из Task 2 загружены')


def evaluate_identification_rate(model, query_dir, distractor_dir, query_csv, distractor_csv,
                                  fpr_values=(0.5, 0.2, 0.1, 0.05)):
    query_df_local = pd.read_csv(query_csv)
    distractor_df_local = pd.read_csv(distractor_csv)

    query_img_names = query_df_local['image_id'].tolist()
    query_paths = [os.path.join(query_dir, name) for name in query_img_names]
    distractor_paths = [os.path.join(distractor_dir, name) for name in distractor_df_local['image_id'].tolist()]

    query_dict = query_df_local.groupby('identity')['image_id'].apply(list).to_dict()

    query_embeddings = compute_embeddings(model, query_paths)
    distractor_embeddings = compute_embeddings(model, distractor_paths)

    cos_pos = compute_cosine_query_pos(query_dict, query_img_names, query_embeddings)
    cos_neg = compute_cosine_query_neg(query_dict, query_img_names, query_embeddings)
    cos_dist = compute_cosine_query_distractors(query_embeddings, distractor_embeddings)

    results = {}
    for fpr in fpr_values:
        thr, tpr = compute_ir(cos_pos, cos_neg, cos_dist, fpr=fpr)
        results[fpr] = (thr, tpr)
        print(f'FPR={fpr:.2f} | threshold={thr:.4f} | TPR={tpr:.4f}')
    return results


print('CE baseline:')
ce_results = evaluate_identification_rate(ce_backbone, 'data/aligned_query/images', 'data/aligned_distractors/images',
                                           'data/aligned_query/aligned_dataset.csv',
                                           'data/aligned_distractors/aligned_dataset.csv')
print()
print('ArcFace:')
arcface_results = evaluate_identification_rate(arcface_backbone, 'data/aligned_query/images', 'data/aligned_distractors/images',
                                                'data/aligned_query/aligned_dataset.csv',
                                                'data/aligned_distractors/aligned_dataset.csv')


CE и ArcFace backbone из Task 2 загружены
CE baseline:
FPR=0.50 | threshold=0.0040 | TPR=0.9889
FPR=0.20 | threshold=0.3178 | TPR=0.8778
FPR=0.10 | threshold=0.4635 | TPR=0.7000
FPR=0.05 | threshold=0.5690 | TPR=0.5667

ArcFace:
FPR=0.50 | threshold=0.0002 | TPR=0.9333
FPR=0.20 | threshold=0.0660 | TPR=0.8111
FPR=0.10 | threshold=0.1070 | TPR=0.7333
FPR=0.05 | threshold=0.1469 | TPR=0.6111


## 5. Сравнение с Triplet Loss

Для полноты картины добавляем в сравнение модель, обученную на Triplet Loss (`Dop2_TripletLoss.ipynb`) —
тот же backbone (EfficientNet-B0), но без классификационной головы: эмбеддинги обучены напрямую
через `TripletMarginWithDistanceLoss` на тройках (anchor, positive, negative). Оцениваем её на тех же
query/distractors, что и CE/ArcFace — так все три подхода сравниваются по единой шкале, а не по разным
метрикам (в Dop2 использовалась упрощённая verification accuracy, здесь — та же строгая TPR@FPR).

In [13]:
TASK_TRIPLET_INPUT_DIR = '/kaggle/input/notebooks/iuliiaburmistrova/notebook73e70c8afe'

class TripletEmbeddingModel(nn.Module):
    def __init__(self, embedding_size=EMBEDDING_SIZE):
        super().__init__()
        backbone = models.efficientnet_b0(weights=None)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Linear(in_features, embedding_size)
        self.backbone = backbone
        self.norm = nn.BatchNorm1d(embedding_size, affine=False)

    def forward(self, x):
        emb = self.backbone(x)
        emb = self.norm(emb)
        return F.normalize(emb, dim=1)

triplet_model = TripletEmbeddingModel().to(DEVICE)
triplet_model.load_state_dict(torch.load(
    os.path.join(TASK_TRIPLET_INPUT_DIR, 'checkpoints_triplet', 'best.pth'), map_location=DEVICE))
triplet_model.eval()

print('Triplet:')
triplet_results = evaluate_identification_rate(triplet_model, 'data/aligned_query/images', 'data/aligned_distractors/images',
                                                 'data/aligned_query/aligned_dataset.csv',
                                                 'data/aligned_distractors/aligned_dataset.csv')

Triplet:
FPR=0.50 | threshold=-0.0617 | TPR=0.9889
FPR=0.20 | threshold=0.3009 | TPR=0.7889
FPR=0.10 | threshold=0.4566 | TPR=0.6444
FPR=0.05 | threshold=0.5625 | TPR=0.4333


### Итоговое сравнение трёх подходов (Identification Rate Metric, незнакомые модели личности)

| FPR  | CE     | ArcFace | Triplet |
|------|--------|---------|---------|
| 0.50 | 0.9889 | 0.9333  | 0.9889  |
| 0.20 | 0.8778 | 0.8111  | 0.7889  |
| 0.10 | 0.7000 | 0.7333  | 0.6444  |
| 0.05 | 0.5667 | 0.6111  | 0.4333  |

На практически значимых (строгих) порогах FPR ArcFace устойчиво лидирует среди всех трёх подходов.
Triplet Loss с простым случайным выбором negative-примеров (без hard mining) показал наименьшую
устойчивость именно на строгих порогах — вероятно, из-за недостатка "трудных" троек в процессе
обучения, о чём и предупреждает теория (см. обсуждение batch hard mining в задании). Это ещё раз
подтверждает: margin-based подход (ArcFace) даёт наиболее сбалансированное качество эмбеддингов для
задачи распознавания лиц среди рассмотренных методов.